# Lab 09: Medallion Architecture (Bronze → Silver → Gold)

The Medallion Architecture is a data engineering approach that organises data into three layers, progressively improving its quality and usability.

- **Bronze Layer (Raw Data):** Stores data as received from the original sources, with little or no transformation. It preserves the original data for future processing.
- **Silver Layer (Cleaned Data):** Contains data that has been cleaned, validated and transformed. Typical operations include removing duplicates, handling missing values and standardising formats.
- **Gold Layer (Business-Ready Data):** Contains highly refined data, often aggregated and organised for specific analytical purposes, such as dashboards, reporting and business intelligence.

The data flows progressively through these three layers: Bronze → Silver → Gold.

<div style="text-align: center;">
  <img src="https://docs.databricks.com/aws/en/assets/images/medallion-architecture-15e2d57ad70d28b1701dda4f7271d862.png" width="700">
</div>

> **In summary:** Bronze stores the data, Silver improves its quality, and Gold makes it ready for analysis and decision-making.

## 1. Notebook configuration and data upload

You are building a small, reliable batch pipeline from `orders_day1.csv`. We will preserve raw data in the bronze layer then quarantine invalid records, keep the latest valid order in the Silver layer, and create a Gold sales summary.

> Remember to create the volume `bda_week9` if you don't want to replace the `DATA_DIR` path, otherwise change it to the volume of databricks you are using.

In [0]:
from pyspark.sql import functions as F, Window
import re

# Data volume where the lab CSVs live (shared with Day 2)
DATA_DIR = "/Volumes/workspace/default/bda_week9"

# Resolving catalog and schema live, so the table names follow wherever the notebook runs
CATALOG = spark.sql("SELECT current_catalog() AS c").first()["c"]
SCHEMA = spark.sql("SELECT current_schema() AS s").first()["s"]

# Table family shared by both practical days (bda09__...), so Day 2 reuses the tables Day 1 creates
PREFIX = "bda09_"
def table_name(suffix):
    return f"`{CATALOG}`.`{SCHEMA}`.`{PREFIX}{suffix}`"
BRONZE = table_name("bronze_orders")
SILVER = table_name("silver_orders")
GOLD = table_name("gold_sales")
QUARANTINE = table_name("quarantine_orders")

print("Data directory:", DATA_DIR)
print("Tables:", BRONZE, SILVER, GOLD, QUARANTINE)


## 2. Reading the file and inspecting the contents of the tables.

As the purpose of this lab is to provide you with capabilities to create pipelines to diagnose the quality of the data and to have it prepared for abalysis we will read the data with the option `inferSchema=false` to ensure that all CSV fields are read as **string**. This helps preserve problematic raw values until we deliberately validate them. 

We will print the schema, show the records, and inspect duplicate order IDs.

In [0]:
# Loading the orders of day1
raw = (spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{DATA_DIR}/orders_day1.csv"))

# Checking the schema of the dataset
print("Schema of the dataset!")
raw.printSchema()

# Printing the number of rows in the dataset
print("CSV rows:", raw.count())
# Displaying the dataset
raw.orderBy("order_id").show(truncate=False)

# Checking if there are duplicated rows
raw.groupBy("order_id").count().filter(F.col("count") > 1).show()

## 3. Persisting the Bronze Delta table 

The **Bronze layer is the entry point of the medallion pipeline**. Its job is to land the data exactly as it arrived — duplicates, invalid values, and all — so we always keep an untouched historical copy to fall back on if anything goes wrong downstream.

We therefore write the raw DataFrame to a managed Delta table **without any cleaning or filtering**. All the columns read from `orders_day1.csv` are stored as-is (as strings), which preserves problematic values until we deliberately validate them in the Silver layer.

The only thing we add is **lineage metadata**, three audit columns attached to every row so we can always trace where the data came from and when it was loaded:

- `_source_file` — the path of the CSV file the row came from, taken from Spark's hidden `_metadata.file_path` column.
- `_ingested_at` — a load timestamp (`current_timestamp()`), recording **when** the row entered the pipeline.
- `batch_id` — a label identifying the delivery this batch belongs to (`"day1"`), which becomes essential later when we process multiple files.

We persist the result with `mode("overwrite")` and `overwriteSchema = true`, making the notebook idempotent: re-running it rebuilds Bronze from scratch with a fresh, consistent schema rather than accumulating duplicates on every execution.

> **Why this matters:** because Bronze is the immutable record of arrival, it is the only layer we can rebuild Silver and Gold from, if a bug is later found in our validation logic.

In [0]:
# Adding lineage metadata to the raw data before persisting it
bronze_day1 = (raw
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("batch_id", F.lit("day1")))

# Writing the bronze Delta table with overwrite mode
(bronze_day1.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(BRONZE))

# Reading the table back and checking the number of rows
bronze = spark.table(BRONZE)
print("Bronze row count:", bronze.count())


## 4. Building a Silver table — the cleaned, validated layer

The **Silver layer is where data quality is enforced**. Bronze deliberately kept every raw row as a string, but analysis needs trustworthy, correctly typed data. Silver's job is to turn the raw arrival record into exactly that: one clean, typed, deduplicated row per real order.

This is done in four stages, and each stage answers one question about the data:

**A. Convert types.** The raw values are still strings, so `try_cast` converts them to proper `INT`, `DECIMAL`, `DATE`, and `TIMESTAMP` columns. This is our first quality check: unlike a normal `CAST`, `try_cast` never crashes the job — a value like `"three"` simply becomes NULL, flagging itself as suspicious rather than killing the pipeline. Watch for this in the output.

**B. Validate each row** against a set of business rules. You complete the two TODO rules (`ids_ok` and `numbers_ok`); the category and date rules are provided. The result is a single boolean condition per row that decides its fate: a row either passes **all** rules or it is invalid. One subtlety: a comparison involving NULL evaluates to NULL, not `False`, so an invalid row could silently match neither the valid nor the invalid filter. Wrapping the condition in `coalesce(..., False)` forces those rows to count as invalid — nothing disappears without explanation.

**C. Quarantine, don't discard.** Rows that fail validation are written to a separate `bda09__quarantine_orders` table together with an `error_reason`, so bad data can be inspected, fixed at the source, and reprocessed later. This is what makes the pipeline *diagnosable* rather than a black box that drops rows silently.

**D. Deduplicate.** The same order can arrive more than once (Bronze showed duplicate order IDs). Using `row_number()` over a window ordered by `updated_at`, we keep only the **latest valid version** of each `order_id` — so Silver ends up as the current, authoritative state of every order.

> **Success criteria for this lab:** your validation should send **3 records** to quarantine, and Silver should contain **16 unique valid orders**. If those counts don't match, inspect the intermediate tables — that is exactly what they are for.

In [0]:
# Stage 1: Convert types

# Converting raw string fields to usable Spark types
typed = (bronze
    .withColumn("quantity_num", F.expr("try_cast(quantity AS INT)"))
    .withColumn("unit_price_num", F.expr("try_cast(unit_price AS DECIMAL(10,2))"))
    .withColumn("order_date_parsed", F.expr("try_cast(order_date AS DATE)"))
    .withColumn("updated_at_parsed", F.expr("try_cast(updated_at AS TIMESTAMP)"))
)
typed.select("order_id", "customer_id", "quantity", "quantity_num", "unit_price", "unit_price_num").show(25, truncate=False)


In [0]:
# Stage 2: Validating each row

'''
Complete the two reusable business rules using F.col(...)
> Hint: isNotNull(), F.F.length(F.trim(...)) > 0, and numeric comparisons
'''
ids_ok = None       # TODO: order_id + customer_id must be present and non-empty
numbers_ok = None   # TODO: quantity_num and unit_price_num must exist and be > 0

valid_condition = (
    ids_ok
    & F.col("product_id").isNotNull()
    & F.col("category").isin("Electronics", "Books", "Clothing")
    & numbers_ok
    & F.col("order_date_parsed").isNotNull()
    & F.col("updated_at_parsed").isNotNull()
)

valid_condition = F.coalesce(valid_condition, F.lit(False))

print("Valid rows before dedup:", typed.filter(valid_condition).count())
print("Invalid rows (quarantine):", typed.filter(~valid_condition).count())

In [0]:
# Stage 3. Sending rows to quarentine
valid_rows = typed.filter(valid_condition)
invalid_rows = (typed.filter(~valid_condition)
    .withColumn("error_reason",
        F.when(F.col("customer_id").isNull() | (F.length(F.trim("customer_id")) == 0), "missing customer ID")
         .when(F.col("quantity_num").isNull(), "quantity cannot be converted to integer")
         .when(F.col("quantity_num") <= 0, "non-positive quantity")
         .otherwise("other validation failure"))
)
invalid_rows.select("order_id", "customer_id", "quantity", "error_reason").show(truncate=False)
invalid_rows.select(*bronze.columns, "error_reason").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE)
print("Valid raw rows before deduplication:", valid_rows.count())
print("Quarantined rows:", invalid_rows.count())


In [0]:
# Stage 4. Deduplicate

# Keep only the newest valid row for each order_id
latest_window = Window.partitionBy("order_id").orderBy(F.col("updated_at_parsed").desc())

# TODO: assign row_number() over latest_window, filter number == 1, then drop it
latest = None

# These are the required columns and types for the Silver Delta table
silver = latest.select(
    "order_id", "customer_id", "product_id", "category",
    F.col("unit_price_num").alias("unit_price"),
    F.col("quantity_num").alias("quantity"),
    F.col("order_date_parsed").alias("order_date"),
    F.col("updated_at_parsed").alias("updated_at"),
    (F.col("unit_price_num") * F.col("quantity_num")).cast("decimal(12,2)").alias("line_total"),
)
(silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(SILVER))
print("Silver row count:", spark.table(SILVER).count())

## 5. Gold table (the business-ready layer): Sales by category

The **Gold layer is the face of the pipeline** — it is what analysts, dashboards, and reports actually query. While Bronze and Silver are engineering layers, Gold answers a *business question* directly, in this lab: **how much revenue did each product category generate, and how many orders drove it?**

Because Silver is already typed, validated, and deduplicated, building table in the Gold layer is a single aggregation: group by `category`, count the rows as `order_count`, and sum `line_total` as `revenue`. Every value in the Gold table inherits the quality guarantees of the Silver — this is the payoff of the medallion structure. Your task is to write that aggregation (Spark SQL or PySpark), using `COUNT(*)` and `SUM(line_total)`, and persist it as a Delta table so it can back reporting tools.

Finally, compute the **grand total revenue** as a single sanity-check number, and optionally a small bar chart of revenue by category.

In [0]:
# Using PySpark
gold = #TODO

gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)
spark.table(GOLD).orderBy("category").show(truncate=False)
print("Total revenue:", spark.table(GOLD).agg(F.sum("revenue")).first()[0])

In [0]:
# Using SparkSQL
gold = #TODO

(gold.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD))

spark.table(GOLD).show(truncate=False)
print("Total revenue:", spark.table(GOLD).agg(F.sum("revenue")).first()[0])

**Questions to reflect on before submitting:**

1. *Why is the Bronze count higher than the Silver count?* Think about what each number represents — where did the difference go, and where can you find every “missing” row accounted for with a reason?
2. *Would it be safe to build Gold directly from Bronze?* Consider what `SUM(line_total)` would mean if the same order were counted twice, if quantities were negative, or if prices were still strings.

**Answers to the reflection questions**

**1. Why is the Bronze count higher than the Silver count?**

Bronze is the raw arrival log: it keeps **all 20 rows** from `orders_day1.csv` exactly as they arrived. Silver keeps exactly one clean, valid, current row per real order, so **4 rows disappear** between the layers — and each one is accounted for somewhere with an explanation:

* **3 rows failed validation** and were quarantined: order **1017** (missing customer ID), **1018** (non-positive quantity) and **1019** (quantity `"three"` could not be cast). All three sit in `bda09__quarantine_orders`, each with its `error_reason` — nothing was silently dropped.
* **1 row was collapsed by deduplication**: order **1003** arrived twice, so `row_number()` kept only the newer `updated_at` version and dropped the older copy. It was a perfectly valid row, so it is *not* in quarantine — but both copies remain in Bronze, and the pre-dedup state stays inspectable in Silver's Delta history.

So 20 − 3 quarantined − 1 duplicate = **16 unique valid orders**, and every "missing" row can be found again, with a reason attached.

**2. Would it be safe to build Gold directly from Bronze?**

No — every Gold number would inherit Bronze's quality problems:

* **Duplicates**: order 1003's revenue (40.00) would be counted **twice**, inflating both `order_count` and `SUM(line_total)`.
* **Invalid rows**: the negative quantity of order 1018 (−2 × 45.00) would *subtract* 90.00 from revenue, and 1019's `"three"` would break the calculation.
* **String types**: in Bronze `unit_price` and `quantity` are still strings, so `line_total` cannot even be computed — a plain `CAST` would crash on `"three"` (which is exactly why Silver uses `try_cast`).

Gold answers a business question, and business answers need trustworthy numbers. Because Silver guarantees one typed, validated, deduplicated row per order, the Gold aggregation reduces to a single `GROUP BY` — each layer of the medallion inherits the quality of the one before it.


### Extra challenge: enriching orders with a product lookup

Real pipelines rarely stop at cleaned fact rows — the next step is **enrichment**: attaching descriptive attributes from a reference table (a *dimension*) to the facts. This is how a bare `product_id` becomes a product name and an authoritative category, and how Gold-level reports can show human-readable labels instead of codes.

`products.csv` (already uploaded to the `bda_week9` volume) is such a lookup table: exactly one row per `product_id`, with `product_name` and `category`. Your task is to join it to the clean orders in Silver using `product_id` as the key, then interrogate the result — because enrichment is also a *quality check in disguise*:

- **Referential integrity:** what happens if an incoming order references a `product_id` that is not in the lookup? Which join keeps that orphan order visible for diagnosis, and which one silently hides it from your results?
- **Category trust:** compare the `category` recorded on each order with the official `product_category` from the lookup. Do they always agree? If not, which source would you trust — and what does that imply for the Gold table you just built from the order-side category?

Try **both an inner join and a left join** and compare the row counts. Before running them, predict which count will be smaller, and why.

In [0]:
# 1. Loading the product lookup table from the volume, read as strings like the orders
products = (spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{DATA_DIR}/products.csv"))

# 2. Renaming the lookup's category so both categories can coexist after the join
products = # TODO

orders = spark.table(SILVER)

# 3. Two flavours of the join: inner keeps only orders with a matching product, left keeps every clean order even when the lookup has no match
joined_inner = # TODO
joined_left = # TODO

print("Clean orders in Silver:", orders.count())
print("Inner join rows (product found):", joined_inner.count())
print("Left join rows (all orders kept):", joined_left.count())

# 4. Orphan orders: product_id not in the lookup -> product_category is NULL after the left join
print("\nOrders with an unknown product_id (visible only thanks to the LEFT join):")
joined_left.filter(F.col("product_category").isNull()).show(truncate=False)

# 5. Category disagreements between what the order claims and what the lookup says
print("Orders whose category disagrees with the official product category:")
joined_inner.filter(F.col("category") != F.col("product_category")).show(truncate=False)

# Day 2 — New Batches, MERGE and Time Travel

The first practical class built the pipeline from a single delivery: overwrite Bronze, rebuild Silver, aggregate Gold. Day 2 moves to the harder, more realistic problem — **maintaining Silver as new data keeps arriving**. `orders_day2.csv` brings several kinds of events at once: brand-new orders, late corrections to existing ones, a repeated delivery, and one *stale* update whose `updated_at` is older than the row already sitting in Silver.

Two Delta Lake capabilities make this possible:

- **`MERGE`** (an *upsert*): update matching rows and insert non-matching rows in a single atomic statement, keeping Silver the current state of every order.
- **Time travel**: every change to a Delta table is versioned, so we can still read Silver exactly as it was before the merge — no backup restore needed.


In [0]:
# Checking if the silver table of the previous class is created or not.
try:
    day1_silver_count = spark.table(SILVER).count()
except Exception as exc:
    raise RuntimeError("Silver table not found. Complete Practical 1 first, or follow 'If you missed Practical 1' in the README") from exc
print("Silver before new batch:", day1_silver_count)


## 6. Ingest the second delivery

Bronze stays an append-only arrival log, so the second delivery lands exactly like the first: every column read as a string, plus the same three lineage columns — `_source_file`, `_ingested_at`, and `batch_id = "day2"`. Those lineage columns are what let you tell Day 1 and Day 2 rows apart later: one Bronze table, two batches, no ambiguity.

One crucial difference from Day 1: this write is an **`append`**, not an `overwrite`. Bronze must accumulate history — overwriting would erase the Day 1 arrival record. But appending makes re-runs dangerous (you would ingest the same file twice), so the provided guard first checks whether `batch_id = "day2"` is already present and skips the append if so. This ensures **idempotence** in the data.

We will inspect the eight incoming records and identify the repeated `order_id`.

In [0]:
# Reading the Day 2 delivery exactly like Day 1: header row, every column as a string
new_raw = (spark.read.option("header","true").option("inferSchema","false")
    .csv(f"{DATA_DIR}/orders_day2.csv"))
print("Day 2 file rows:", new_raw.count())

# Checking for repeated order IDs in the delivery: the reason Task 2 must deduplicate before merging
new_raw.groupBy("order_id").count().filter("count > 1").show()

# Idempotence guard: append the batch only if it is not already in Bronze, so rerunning the notebook never ingests the same file twice
has_day2 = spark.table(BRONZE).filter(F.col("batch_id") == "day2").limit(1).count() > 0
if not has_day2:
    (new_raw
        # Same three lineage columns as Day 1, now with batch_id = "day2"
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("batch_id", F.lit("day2"))
        .write.format("delta").mode("append").saveAsTable(BRONZE))
    print("Appended day2 to Bronze")
else:
    print("Day 2 already ingested: skipping duplicate batch")
print("Bronze total:", spark.table(BRONZE).count())

## 7. Validate, stage one update per order, and quarantine

Day 2 applies the exact same quality standards as the first practical class — the helper below deliberately reuses the same rules (`try_cast` typing, the same business conditions, `row_number()` deduplication) so you can focus on the new topic: Delta Lake. Bad rows still go to quarantine with a reason, so nothing vanishes.

The `row_number()` step is not just hygiene here — it is a **hard requirement for `MERGE`**. For every target row, Delta must know exactly *one* source row to apply. If two Day 2 source rows match the same Silver row, the `MERGE` fails with an error rather than guessing which update wins. That is why the repeated `order_id` you spotted in Task 1 must be collapsed to its newest version *before* the merge.

After running the helper, note the three numbers it prints: how many raw rows arrived, how many were rejected, and how many **unique valid** update records remain to be staged.

In [0]:
# Same quality standards as the previous class.

def add_error_reason(df):
    """Same error reasons as in Practical 1."""
    return df.withColumn("error_reason",
        F.when(F.col("customer_id").isNull() | (F.length(F.trim("customer_id")) == 0), "missing customer ID")
         .when(F.col("quantity_num").isNull(), "quantity cannot be converted to integer")
         .when(F.col("quantity_num") <= 0, "non-positive quantity")
         .otherwise("other validation failure"))

def prepare_valid_orders(raw_df):
    """Return valid, one-row-per-order records with the exact Silver schema."""
    # Stage 1: same try_cast typing as Practical 1 ("three" becomes NULL, not a crash)
    typed = (raw_df
        .withColumn("quantity_num", F.expr("try_cast(quantity AS INT)"))
        .withColumn("unit_price_num", F.expr("try_cast(unit_price AS DECIMAL(10,2))"))
        .withColumn("order_date_parsed", F.expr("try_cast(order_date AS DATE)"))
        .withColumn("updated_at_parsed", F.expr("try_cast(updated_at AS TIMESTAMP)")))
    # Stage 2: same business rules, with coalesce so NULL comparisons count as invalid (no silent drops)
    valid = F.coalesce(
        F.col("order_id").isNotNull() & (F.length(F.trim("order_id")) > 0)
        & F.col("customer_id").isNotNull() & (F.length(F.trim("customer_id")) > 0)
        & F.col("product_id").isNotNull()
        & F.col("category").isin("Electronics", "Books", "Clothing")
        & F.col("quantity_num").isNotNull() & (F.col("quantity_num") > 0)
        & F.col("unit_price_num").isNotNull() & (F.col("unit_price_num") > 0)
        & F.col("order_date_parsed").isNotNull()
        & F.col("updated_at_parsed").isNotNull(),
        F.lit(False))   # NULL counts as invalid, so no row can disappear
    valid_rows = typed.filter(valid)
    bad_rows = typed.filter(~valid)
    # Stage 3: one row per order, newest first — a hard requirement for MERGE,
    # which fails if two source rows match the same target row
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at_parsed").desc())
    latest = (valid_rows.withColumn("row_number", F.row_number().over(w))
        .filter(F.col("row_number")==1).drop("row_number"))
    # Stage 4: the exact Silver schema, including the derived line_total
    clean = latest.select(
        "order_id", "customer_id", "product_id", "category",
        F.col("unit_price_num").alias("unit_price"),
        F.col("quantity_num").alias("quantity"),
        F.col("order_date_parsed").alias("order_date"),
        F.col("updated_at_parsed").alias("updated_at"),
        (F.col("unit_price_num") * F.col("quantity_num")).cast("decimal(12,2)").alias("line_total"))
    return clean, bad_rows, valid_rows

In [0]:
# Staging only the Day 2 batch from Bronze: the Day 1 rows stay untouched
incoming = spark.table(BRONZE).filter(F.col("batch_id") == "day2")
staged, rejected, valid_raw = prepare_valid_orders(incoming)

# The three numbers to note: arrivals, rejects, and unique valid updates left to merge
print("Incoming records:", incoming.count())
print("Rejected records:", rejected.count())
print("Valid records before deduplication:", valid_raw.count())
print("Distinct staged orders:", staged.count())
staged.orderBy("order_id").show(truncate=False)

# Exposing the staged updates as a temp view
staged.createOrReplaceTempView("bda09_incoming_updates")

In [0]:
# 1. Attaching the same error reasons as the previous class, so every reject comes with an explanation
rejected_with_reason = add_error_reason(rejected)

# 2. Bronze columns + error_reason: the exact quarantine schema from the previous class - (the *_num / *_parsed helper columns are dropped, the raw string values are kept)
bronze_columns = spark.table(BRONZE).columns
quarantine_day2 = # TODO
print("Day 2 rejected records quarantined:", quarantine_day2.count())

# 3. Same append-once guard as the Bronze ingest, so rerunning never duplicates the rejects
already_quarantined = # TODO
if not already_quarantined:
    quarantine_day2.write.format("delta").mode("append").saveAsTable(QUARANTINE)
    print("Appended Day 2 rejects to quarantine")
else:
    print("Day 2 rejects already in quarantine: skipping")

# Both batches side by side, every reject with its reason
(spark.table(QUARANTINE).select("order_id", "batch_id", "quantity", "error_reason")
    .orderBy("batch_id", "order_id").show(truncate=False))

## 8. Use Delta MERGE to maintain Silver

`MERGE INTO ... USING ... ON` is Delta's upsert: in one atomic statement it can update target rows that match the source and insert the ones that don't. Your job is to write the *policy* for each case:

- **`WHEN MATCHED`** — the order already exists in Silver, so this row is a correction. Apply it **only when** `s.updated_at > t.updated_at`; without that condition you would overwrite newer truth with the stale update hiding in the batch.
- **`WHEN NOT MATCHED`** — the order has never been seen before, so `INSERT` it as a new row.

Because source and target have identical columns, `UPDATE SET *` and `INSERT *` save you from listing every column by hand. The timestamp guard is what makes the merge *safe*, not just convenient.

Before changing anything, the provided cell captures the **pre-merge baseline**: the last Silver version and its total revenue. It finds that version from the table history instead of assuming version 0, so the cell still works on re-runs when the MERGE itself has created later versions.

In [0]:
# Capturing the pre-merge baseline: the last Silver version written by Practical 1
# (CREATE / REPLACE / WRITE). Using plain max(version) would point to the
# post-MERGE version when this notebook is rerun.
history_before = spark.sql(f"DESCRIBE HISTORY {SILVER}")
PRE_MERGE_VERSION = (history_before
    .filter(F.col("operation").rlike("CREATE|REPLACE|WRITE"))
    .agg(F.max("version")).first()[0])
print("Version before MERGE (Day 1 Silver):", PRE_MERGE_VERSION)

# The Day 1 revenue, read via time travel from that version
print("Silver revenue before:",
      spark.sql(f"SELECT SUM(line_total) AS r FROM {SILVER} VERSION AS OF {PRE_MERGE_VERSION}").first()["r"])

In [0]:
MERGE_SQL = f"""
MERGE INTO {SILVER} AS t
USING bda09_incoming_updates AS s
ON t.order_id = s.order_id
-- TODO add the WHEN MATCHED condition and UPDATE SET *
-- TODO add WHEN NOT MATCHED THEN INSERT *
"""
spark.sql(MERGE_SQL)
print("Silver rows after MERGE:", spark.table(SILVER).count())

# Sanity check: order 1005 keeps quantity 3 — the stale incoming value 2 was skipped
spark.table(SILVER).filter(F.col("order_id") == "1005").show(truncate=False)

## 9. Time travel and refresh Gold

Every write to a Delta table — the original create, the MERGE, any future overwrite — adds a **version** to the table's history. `DESCRIBE HISTORY` makes that timeline visible, and each version can itself be queried like a table: `FROM ... VERSION AS OF <n>` reads the snapshot exactly as it stood at that point.

That is what the pre-merge baseline was for: compare Silver **before** the MERGE (via `PRE_MERGE_VERSION`) with its latest state — new orders inserted, corrections applied, stale update ignored — side by side, with no backup restore and no second table.

Then close the loop: Gold was built from Day 1 Silver, so it is now stale. Recalculate it from the **current** Silver snapshot with the same aggregation as in the first practical class, so the whole medallion is consistent again end-to-end.

In [0]:
# The full version timeline of Silver: every write (CREATE, WRITE, MERGE) adds one version
history = spark.sql(f"DESCRIBE HISTORY {SILVER}")
history.select("version", "timestamp", "operation").show(truncate=False)

# Comparing the pre-merge snapshot with the current state, side by side, without any backup
old_rows = spark.sql(f"SELECT * FROM {SILVER} VERSION AS OF {PRE_MERGE_VERSION}")
print("Rows in pre-merge version:", old_rows.count())
print("Revenue in pre-merge version:", old_rows.agg(F.sum("line_total")).first()[0])
print("Current revenue:", spark.table(SILVER).agg(F.sum("line_total")).first()[0])

In [0]:
new_gold = # TODO

new_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD)
spark.table(GOLD).orderBy("category").show(truncate=False)
print("Total revenue:", spark.table(GOLD).agg(F.sum("revenue")).first()[0])

**Questions to reflect on:**

1. *Why must the update batch be deduplicated before the MERGE?* What does Delta do if two source rows claim to update the same target row?
2. *Why should the stale update to order `1005` be ignored?* Which clause protected it, and what would the data look like without that condition?
3. *How is Delta time travel different from a backup?* Think about what a version actually is, how cheap it is to create and query, and whether it protects you from the table itself being deleted.

**Answers to the reflection questions**

**1. Why must the update batch be deduplicated before the MERGE?**

`MERGE` must know, for every target row, exactly **one** source row to apply. If two source rows match the same `order_id` in Silver, Delta refuses to guess which one wins: the statement fails with `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE` instead of silently picking a winner (or, worse, updating the same row twice). The Day 2 file contained order **1021** twice — the repeated delivery — so without the `row_number()` stage collapsing it to its newest `updated_at` version, the whole merge would have errored and *no* update would have been applied. Deduplicating first turns an all-or-nothing failure into a deterministic, atomic upsert.

**2. Why should the stale update to order 1005 be ignored?**

Silver already holds a **newer** truth for 1005: quantity 3, written by Day 1 at `2026-10-28 09:16`. The Day 2 row carries quantity 2 with an **older** `updated_at` (`2026-10-28 08:30`) — a late-arriving event that was already superseded. The protecting clause is `WHEN MATCHED AND s.updated_at > t.updated_at`: it only lets a correction through when it is strictly newer than what Silver holds, so 1005 correctly stayed at quantity 3 (and `line_total` 30.00) in the run above. Without that condition, `UPDATE SET *` would blindly apply every match and drag Silver **back in time** — quantity 3 would become 2, revenue would drop, and even `updated_at` itself would be overwritten with the older timestamp, corrupting the audit trail for every future merge.

**3. How is Delta time travel different from a backup?**

A **version** is not a copy of the table — it is a commit in the table's own history: an entry in the transaction log pointing at immutable data files already written. So it is created **automatically on every write** (costing essentially nothing), lives **inside** the table, and is queryable **in place** with `VERSION AS OF` — that is how this notebook compared the pre-merge snapshot (16 rows, revenue 686.00) with the current state (19 rows, 849.00) side by side without any restore. A backup, by contrast, must be explicitly taken, stored, and restored — slow and costly. But that convenience has limits: time travel protects against **bad writes**, not against **losing the table itself** — `DROP TABLE` deletes the log and the data together, so no version can bring it back. And versions are not kept forever: retention settings and `VACUUM` eventually clean old files, so time travel is a short-horizon safety net, not an archival strategy.


### Extra challenges: idempotence and constraints

**A. Idempotence — rerunning the whole pipeline.** A production notebook is run repeatedly, so it must be safe to run twice. Run the notebook again and audit every layer: does Bronze grow? Does quarantine grow? Do Silver's row count or revenue change? Explain which steps are idempotent and *why* — and note that the MERGE has no explicit guard the way the Bronze append does; think about what happens when the same updates are applied a second time. Finally, find order `1005` and verify its quantity remains **3**, despite the stale incoming value **2**.

**B. CHECK constraint — prevention instead of quarantine.** Quarantine handles bad rows *after* they arrive; a table **constraint** stops them from ever entering. Add a Delta CHECK constraint so Silver can never contain a non-positive quantity, then run the provided test insert of a zero-quantity order. What happens — and what does that reactive-vs-preventive contrast tell you about when to use each mechanism?

**Submit:** updated Gold results, a history table (version/operation), and answers to the three interpretation questions.

In [0]:
# Write the statement that adds a constraint named positive_quantity requiring quantity > 0
# Hint: ALTER TABLE {SILVER} ADD CONSTRAINT ... CHECK (...)
CONSTRAINT_SQL = # TODO

# Adding the constraint only once (Delta stores it as the table property delta.constraints.<name>)
exists = (spark.sql(f"SHOW TBLPROPERTIES {SILVER}")
          .filter(F.col("key") == "delta.constraints.positive_quantity").count() > 0)
if not exists:
    spark.sql(CONSTRAINT_SQL)
    print("Constraint positive_quantity added to Silver")
else:
    print("Constraint already present: skipping")

# Test: the zero-quantity order must now be rejected by Delta itself, not by our code
try:
    spark.sql(f"""INSERT INTO {SILVER} VALUES ('9999', 'C999', 'P100', 'Electronics', 25.00, 0,
                  DATE'2026-10-30', TIMESTAMP'2026-10-30 12:00:00', 0.00)""")
    print("Insert succeeded: the constraint is missing or wrong. Removing the test row again.")
    spark.sql(f"DELETE FROM {SILVER} WHERE order_id = '9999'")
except Exception as exc:
    print("Insert rejected by Delta:", str(exc).splitlines()[0][:200])